# HGCTM Revision — Strong Geographic Generalisation Audit

This notebook is the stronger follow-up to the corrected continent holdout.

It implements four improvements:

1. **One fixed training-only LDA warm start per continent fold.**  
   LDA is fitted once with `random_state = 42` on the five training continents and is reused for all three Pyro seeds.

2. **A direct context-only Dirichlet–Multinomial baseline.**  
   This model uses lithology and age but has no topic mixture. It directly tests whether the topic layer adds predictive information beyond the covariates.

3. **Only directly comparable held-out likelihoods.**  
   HGCTM, context-only DM, and global DM are evaluated with the same complete Dirichlet–Multinomial likelihood. Raw scikit-learn LDA likelihoods are not compared with HGCTM.

4. **A training-only K audit.**  
   Candidate values `K = 5, 6, 7, 8, 9` are evaluated using repeated inner train/validation splits constructed only from the five outer-training continents. This checks whether K=7 remains competitive without using the held-out continent.

LDA is used only for the fixed fold-specific warm start and for post hoc fold-to-full topic-retention analysis.

This remains a **post-selection geographic sensitivity analysis** because the model-development strategy preceded this follow-up. It must not be described as untouched external model selection.

## Required Kaggle inputs

Attach:

1. `Global_Copper_Deposit_Main.xlsx`
2. `Hgctm_stage0.zip`
3. `HGCTM_Phase1B_GLiM_Audit_Outputs.zip`
4. `HGCTM_Prior_and_Lithology_Class_Sensitivity_K7_Outputs.zip`

No new manual data or information is required.

The fourth archive is used only for post hoc alignment with the matching full-cohort HGCTM topics. It is never used to fit or score a held-out fold.

> **Corrected version:** this notebook now defines the fixed LDA seed and K-audit settings, imports `StratifiedShuffleSplit`, and standardizes the continent column. Run all cells from the beginning.


In [ ]:
# Install only missing packages.

import importlib.util
import subprocess
import sys

required = {
    "pyro": "pyro-ppl",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])
else:
    print("All required packages are available.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import time
import warnings
import zipfile

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from torch.nn.functional import softmax

from scipy.optimize import linear_sum_assignment
from scipy.special import gammaln
from scipy.stats import chi2_contingency
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working")
OUT = WORK / "hgctm_strong_geographic_audit"
RUNS_DIR = OUT / "runs"
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"

for folder in [OUT, RUNS_DIR, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# ---------------- EXECUTION SETTINGS ----------------

RUN_MODE = "full"

K = 7
SIGMA_MU = 2.0
SIGMA_LITHO = 0.30
SIGMA_AGE = 0.15
LR = 0.005
MIN_HOLDOUT_N = 30
MIN_MINERALS = 3
MIN_DEPOSITS_PER_FAMILY = 10
TOP_N_FAMILIES = 10

FULL_SEEDS = [42, 123, 7]
PRIMARY_SEED = 42
FIXED_LDA_SEED = 42

# Optional training-only K audit.
RUN_NESTED_K_AUDIT = True
K_CANDIDATES = [5, 6, 7, 8, 9]
K_AUDIT_SPLIT_SEEDS = [101, 202, 303]
K_AUDIT_VALIDATION_FRACTION = 0.20
K_AUDIT_MAX_ITER = 120

if RUN_MODE == "quick":
    ACTIVE_SEEDS = [42]
    N_STEPS = 400
    ANNEAL_END = 120
    PRINT_EVERY = 100
    THETA_INFERENCE_STEPS = 50
    BOOTSTRAP_REPLICATES = 200
elif RUN_MODE == "screening":
    ACTIVE_SEEDS = [42]
    N_STEPS = 2500
    ANNEAL_END = 750
    PRINT_EVERY = 500
    THETA_INFERENCE_STEPS = 150
    BOOTSTRAP_REPLICATES = 500
elif RUN_MODE == "full":
    ACTIVE_SEEDS = FULL_SEEDS
    N_STEPS = 5000
    ANNEAL_END = 1500
    PRINT_EVERY = 500
    THETA_INFERENCE_STEPS = 250
    BOOTSTRAP_REPLICATES = 1000
else:
    raise ValueError("RUN_MODE must be quick, screening, or full.")

EXPECTED_MODEL_N = 1335
EXPECTED_INVALID_COORDINATES = 98
EXPECTED_VALID_N = 1237
EXPECTED_F = 35

AGE_LEVELS = [
    "Cenozoic",
    "Mesozoic",
    "Paleozoic",
    "Precambrian",
    "Unknown",
]

print("Run mode:", RUN_MODE)
print("Seeds:", ACTIVE_SEEDS)
print("Steps:", N_STEPS)
print("Fixed fold-LDA seed:", FIXED_LDA_SEED)
print("Nested K audit:", RUN_NESTED_K_AUDIT)


## 1. Locate and extract the frozen inputs

In [ ]:
def extract_zip_once(zip_path, target):
    target = Path(target)
    marker = target / ".extracted_ok"

    if marker.exists():
        return target

    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)

    marker.write_text(
        datetime.now(timezone.utc).isoformat(),
        encoding="utf-8",
    )
    return target


def find_file(filename):
    roots = [Path("/kaggle/input"), WORK]

    for root in roots:
        if not root.exists():
            continue
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not locate {filename}")


def find_folder_with(required_names, label):
    required_names = set(required_names)
    roots = [Path("/kaggle/input"), WORK]

    for root in roots:
        if not root.exists():
            continue
        for first_name in required_names:
            for path in root.rglob(first_name):
                folder = path.parent
                present = {
                    item.name
                    for item in folder.iterdir()
                    if item.is_file()
                }
                if required_names.issubset(present):
                    return folder

    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name
                        for name in archive.namelist()
                    }
                    if required_names.issubset(basenames):
                        safe_label = re.sub(
                            r"[^a-z0-9]+",
                            "_",
                            label.lower(),
                        ).strip("_")
                        digest = hashlib.sha256(
                            str(zip_path).encode("utf-8")
                        ).hexdigest()[:10]
                        target = WORK / f"{safe_label}_{digest}"
                        extract_zip_once(zip_path, target)

                        for first_name in required_names:
                            matches = list(target.rglob(first_name))
                            for match in matches:
                                folder = match.parent
                                available = {
                                    item.name
                                    for item in folder.iterdir()
                                    if item.is_file()
                                }
                                if required_names.issubset(available):
                                    return folder
            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError(
        f"Could not locate {label}. Required files: "
        f"{sorted(required_names)}"
    )


MAIN_EXCEL = find_file("Global_Copper_Deposit_Main.xlsx")

STAGE0 = find_folder_with(
    {"X_family_copper.csv", "covariates.csv"},
    "HGCTM Stage 0",
)

PHASE1B = find_folder_with(
    {
        "covariates_lithology_A_macrostrat_original.csv",
        "covariates_lithology_B_glim_only.csv",
    },
    "HGCTM Phase 1B",
)

print("Main Excel:", MAIN_EXCEL)
print("Stage 0:", STAGE0)
print("Phase 1B:", PHASE1B)


In [ ]:
def locate_full_reference_runs():
    expected = "macrostrat__original__seed_42.npz"
    roots = [Path("/kaggle/input"), WORK]

    for root in roots:
        if not root.exists():
            continue
        matches = list(root.rglob(expected))
        if matches:
            return matches[0].parent

    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name
                        for name in archive.namelist()
                    }
                    if expected in basenames:
                        digest = hashlib.sha256(
                            str(zip_path).encode("utf-8")
                        ).hexdigest()[:10]
                        target = WORK / f"full_reference_{digest}"
                        extract_zip_once(zip_path, target)
                        matches = list(target.rglob(expected))
                        if matches:
                            return matches[0].parent
            except zipfile.BadZipFile:
                continue

    return None


FULL_REFERENCE_RUNS = locate_full_reference_runs()

if FULL_REFERENCE_RUNS is None:
    print(
        "WARNING: Full-cohort reference runs were not found. "
        "Holdout fitting and scoring will still run, but post hoc "
        "topic-retention comparisons will be skipped."
    )
else:
    print("Full reference runs:", FULL_REFERENCE_RUNS)


## 2. Reconstruct the 1,335-deposit model set

In [ ]:
def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


def canonicalize_index(frame, label):
    frame = frame.copy()
    frame.index = [
        canonical_id(value)
        for value in frame.index
    ]
    frame.index.name = "Mindat_id"

    if frame.index.isna().any():
        raise ValueError(f"{label} contains missing IDs.")
    if frame.index.duplicated().any():
        raise ValueError(f"{label} contains duplicate IDs.")

    return frame


def canonicalize_id_column(frame, label):
    frame = frame.copy()

    if "Mindat_id" not in frame.columns:
        raise KeyError(f"{label} has no Mindat_id column.")

    frame["Mindat_id"] = frame["Mindat_id"].map(canonical_id)

    if frame["Mindat_id"].isna().any():
        raise ValueError(f"{label} contains missing IDs.")
    if frame["Mindat_id"].duplicated().any():
        raise ValueError(f"{label} contains duplicate IDs.")

    return frame


X_raw = canonicalize_index(
    pd.read_csv(
        STAGE0 / "X_family_copper.csv",
        index_col=0,
    ),
    "X_family_copper",
)

cov_stage0 = canonicalize_index(
    pd.read_csv(
        STAGE0 / "covariates.csv",
        index_col=0,
    ),
    "covariates",
)

macro = canonicalize_id_column(
    pd.read_csv(
        PHASE1B
        / "covariates_lithology_A_macrostrat_original.csv"
    ),
    "Macrostrat",
)

glim = canonicalize_id_column(
    pd.read_csv(
        PHASE1B
        / "covariates_lithology_B_glim_only.csv"
    ),
    "GLiM",
)

main = pd.read_excel(
    MAIN_EXCEL,
    sheet_name="Sheet1",
)

main = main.dropna(how="all").copy()
main["Mindat_id"] = main["Mindat_id"].map(canonical_id)
main = main.dropna(subset=["Mindat_id"])
main = main.drop_duplicates(
    subset=["Mindat_id"],
    keep="first",
)

model_mask = X_raw.sum(axis=1) >= MIN_MINERALS
X_pre_family = X_raw.loc[model_mask].copy()

family_keep = (
    (X_pre_family > 0)
    .sum(axis=0)
)
family_keep = family_keep[
    family_keep >= MIN_DEPOSITS_PER_FAMILY
].index.tolist()

X_model = X_pre_family[family_keep].copy()

model_ids = [
    mindat_id
    for mindat_id in X_model.index
    if mindat_id in cov_stage0.index
]

X_model = X_model.loc[model_ids].copy()
cov_model = cov_stage0.loc[model_ids].copy()

if X_model.shape != (EXPECTED_MODEL_N, EXPECTED_F):
    raise ValueError(
        f"Expected ({EXPECTED_MODEL_N}, {EXPECTED_F}), "
        f"found {X_model.shape}."
    )

print("Reconstructed model matrix:", X_model.shape)


## 3. Merge country, corrected lithology statuses, and coordinate validity

In [ ]:
country_fields = main[
    [
        "Mindat_id",
        "Deposit_name",
        "Country",
        "Latitude",
        "Longitude",
    ]
].copy()

macro_fields = macro[
    [
        "Mindat_id",
        "lithology_A_class",
        "lithology_A_status",
    ]
].copy()

glim_fields = glim[
    [
        "Mindat_id",
        "lithology_B_class",
        "lithology_B_status",
    ]
].copy()

audit = cov_model.reset_index().merge(
    country_fields,
    on="Mindat_id",
    how="left",
    suffixes=("_stage0", "_main"),
    validate="one_to_one",
)

audit = audit.merge(
    macro_fields,
    on="Mindat_id",
    how="left",
    validate="one_to_one",
)

audit = audit.merge(
    glim_fields,
    on="Mindat_id",
    how="left",
    validate="one_to_one",
)

# Stage 0 coordinates are the frozen model coordinates.
audit["Latitude"] = audit["Latitude_stage0"]
audit["Longitude"] = audit["Longitude_stage0"]


def valid_coordinate(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False

    try:
        lat = float(lat)
        lon = float(lon)
    except Exception:
        return False

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return False

    if abs(lat) < 1e-12 and abs(lon) < 1e-12:
        return False

    return True


audit["coordinate_valid"] = audit.apply(
    lambda row: valid_coordinate(
        row["Latitude"],
        row["Longitude"],
    ),
    axis=1,
)

if (~audit["coordinate_valid"]).sum() != EXPECTED_INVALID_COORDINATES:
    raise ValueError(
        f"Expected {EXPECTED_INVALID_COORDINATES} invalid coordinates, "
        f"found {(~audit['coordinate_valid']).sum()}."
    )

print("Full audit rows:", len(audit))
print("Coordinate-valid:", audit["coordinate_valid"].sum())
print("Coordinate-missing/invalid:", (~audit["coordinate_valid"]).sum())


## 4. Country-to-continent assignment

Continents are assigned from the curated country field using a frozen mapping. The convention follows a single administrative grouping:

- Russia is grouped with Europe;
- Turkey, Kazakhstan, Georgia, Armenia, Azerbaijan, and Cyprus are grouped with Asia;
- Mexico, Central America, and the Caribbean are grouped with North America.

The one record labelled `Eurasian Plate` is retained as `Unknown`.

In [ ]:
COUNTRY_TO_CONTINENT = {
    "Afghanistan": "Asia",
    "Algeria": "Africa",
    "Angola": "Africa",
    "Argentina": "South America",
    "Armenia": "Asia",
    "Australia": "Oceania",
    "Azerbaijan": "Asia",
    "Belgium": "Europe",
    "Bolivia": "South America",
    "Botswana": "Africa",
    "Brazil": "South America",
    "Bulgaria": "Europe",
    "Canada": "North America",
    "Chile": "South America",
    "China": "Asia",
    "Colombia": "South America",
    "Costa Rica": "North America",
    "Cuba": "North America",
    "Cyprus": "Asia",
    "DR Congo": "Africa",
    "Dominican Republic": "North America",
    "Ecuador": "South America",
    "Egypt": "Africa",
    "Eurasian Plate": "Unknown",
    "Fiji": "Oceania",
    "Finland": "Europe",
    "France": "Europe",
    "Georgia": "Asia",
    "Germany": "Europe",
    "Greece": "Europe",
    "Guatemala": "North America",
    "Haiti": "North America",
    "Hungary": "Europe",
    "India": "Asia",
    "Indonesia": "Asia",
    "Iran": "Asia",
    "Ireland": "Europe",
    "Israel": "Asia",
    "Italy": "Europe",
    "Japan": "Asia",
    "Kazakhstan": "Asia",
    "Kosovo": "Europe",
    "Kyrgyzstan": "Asia",
    "Laos": "Asia",
    "Malaysia": "Asia",
    "Mali": "Africa",
    "Mauritania": "Africa",
    "Mexico": "North America",
    "Mongolia": "Asia",
    "Morocco": "Africa",
    "Myanmar": "Asia",
    "Namibia": "Africa",
    "Niger": "Africa",
    "North Macedonia": "Europe",
    "Norway": "Europe",
    "Oman": "Asia",
    "Pakistan": "Asia",
    "Panama": "North America",
    "Papua New Guinea": "Oceania",
    "Peru": "South America",
    "Philippines": "Asia",
    "Poland": "Europe",
    "Portugal": "Europe",
    "Puerto Rico": "North America",
    "Romania": "Europe",
    "Russia": "Europe",
    "Saudi Arabia": "Asia",
    "Serbia": "Europe",
    "Slovakia": "Europe",
    "Solomon Islands": "Oceania",
    "South Africa": "Africa",
    "Spain": "Europe",
    "Sweden": "Europe",
    "Taiwan": "Asia",
    "Thailand": "Asia",
    "Turkey": "Asia",
    "USA": "North America",
    "Uganda": "Africa",
    "Uzbekistan": "Asia",
    "Vietnam": "Asia",
    "Zambia": "Africa",
    "Zimbabwe": "Africa",
}

audit["Country_clean"] = (
    audit["Country"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

audit["continent_country"] = (
    audit["Country_clean"]
    .map(COUNTRY_TO_CONTINENT)
    .fillna("Unknown")
)

unmapped_countries = sorted(
    set(audit["Country_clean"])
    - set(COUNTRY_TO_CONTINENT)
    - {"Unknown"}
)

if unmapped_countries:
    raise ValueError(
        f"Unmapped countries: {unmapped_countries}"
    )

valid_unknown_continent = audit[
    audit["coordinate_valid"]
    & audit["continent_country"].eq("Unknown")
]

if len(valid_unknown_continent):
    raise ValueError(
        "Coordinate-valid records have unknown continent: "
        f"{valid_unknown_continent[['Mindat_id', 'Country_clean']].to_dict('records')}"
    )

continent_mapping_table = pd.DataFrame(
    sorted(COUNTRY_TO_CONTINENT.items()),
    columns=["Country", "Continent"],
)

continent_mapping_table.to_csv(
    TABLE_DIR / "country_to_continent_mapping.csv",
    index=False,
)

print("Full model-set continent counts:")
print(audit["continent_country"].value_counts().to_string())

print("\nCoordinate-valid continent counts:")
print(
    audit.loc[
        audit["coordinate_valid"],
        "continent_country",
    ].value_counts().to_string()
)


## 5. Unknown-class and coordinate-missing audit

In [ ]:
def normalize_lithology_status(
    class_value,
    status_value,
):
    class_text = (
        "unknown"
        if pd.isna(class_value)
        else str(class_value).strip().lower()
    )
    status_text = (
        ""
        if pd.isna(status_value)
        else str(status_value).strip().lower()
    )

    if class_text == "unknown" or status_text == "unknown":
        return "unknown"
    if class_text == "other" or status_text == "other":
        return "other"
    return "resolved"


audit["Macrostrat_lithology_status"] = audit.apply(
    lambda row: normalize_lithology_status(
        row["lithology_A_class"],
        row["lithology_A_status"],
    ),
    axis=1,
)

audit["GLiM_lithology_status"] = audit.apply(
    lambda row: normalize_lithology_status(
        row["lithology_B_class"],
        row["lithology_B_status"],
    ),
    axis=1,
)

audit["age_status"] = np.where(
    audit["age_category"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .eq("Unknown"),
    "unknown",
    "resolved",
)

audit["coordinate_status"] = np.where(
    audit["coordinate_valid"],
    "valid",
    "missing_or_invalid",
)

audit["Macrostrat_nonresolved"] = (
    audit["Macrostrat_lithology_status"]
    .isin(["other", "unknown"])
)

audit["GLiM_nonresolved"] = (
    audit["GLiM_lithology_status"]
    .isin(["other", "unknown"])
)

audit["age_unknown"] = audit["age_status"].eq("unknown")
audit["coordinate_missing"] = ~audit["coordinate_valid"]

audit.to_csv(
    TABLE_DIR / "full_model_set_unknown_class_audit.csv",
    index=False,
)


def grouped_status_table(
    frame,
    cohort_name,
    grouping_columns,
    status_column,
):
    counts = (
        frame.groupby(
            grouping_columns + [status_column],
            dropna=False,
        )
        .size()
        .rename("count")
        .reset_index()
    )

    totals = (
        frame.groupby(
            grouping_columns,
            dropna=False,
        )
        .size()
        .rename("group_total")
        .reset_index()
    )

    result = counts.merge(
        totals,
        on=grouping_columns,
        how="left",
    )

    result["percent_within_group"] = (
        100
        * result["count"]
        / result["group_total"]
    )

    result["cohort"] = cohort_name
    result["status_variable"] = status_column

    return result


audit_tables = []

cohorts = {
    "full_model_1335": audit.copy(),
    "coordinate_valid_1237": audit[
        audit["coordinate_valid"]
    ].copy(),
}

status_columns = [
    "Macrostrat_lithology_status",
    "GLiM_lithology_status",
    "age_status",
    "coordinate_status",
]

groupings = [
    ["Deposit_type"],
    ["continent_country"],
    ["Deposit_type", "continent_country"],
]

for cohort_name, frame in cohorts.items():
    for status_column in status_columns:
        for grouping in groupings:
            table = grouped_status_table(
                frame,
                cohort_name,
                grouping,
                status_column,
            )
            table["grouping"] = " × ".join(grouping)
            audit_tables.append(table)

unknown_distribution_long = pd.concat(
    audit_tables,
    ignore_index=True,
    sort=False,
)

unknown_distribution_long.to_csv(
    TABLE_DIR / "unknown_classes_by_type_and_continent_long.csv",
    index=False,
)

print(
    unknown_distribution_long[
        (
            unknown_distribution_long["cohort"]
            == "full_model_1335"
        )
        & (
            unknown_distribution_long["grouping"]
            == "continent_country"
        )
        & (
            unknown_distribution_long["status_variable"]
            == "Macrostrat_lithology_status"
        )
    ][
        [
            "continent_country",
            "Macrostrat_lithology_status",
            "count",
            "group_total",
            "percent_within_group",
        ]
    ].round(2).to_string(index=False)
)


## 6. Statistical audit of uneven missingness

In [ ]:
def cramers_v(contingency):
    table = np.asarray(
        contingency,
        dtype=float,
    )

    if table.size == 0:
        return np.nan

    chi2, _, _, _ = chi2_contingency(
        table,
        correction=False,
    )

    n = table.sum()
    r, k = table.shape

    denominator = min(r - 1, k - 1)

    if n <= 0 or denominator <= 0:
        return np.nan

    return float(
        np.sqrt(
            (chi2 / n)
            / denominator
        )
    )


association_rows = []

binary_outcomes = [
    "Macrostrat_nonresolved",
    "GLiM_nonresolved",
    "age_unknown",
    "coordinate_missing",
]

group_variables = [
    "Deposit_type",
    "continent_country",
]

for cohort_name, frame in cohorts.items():
    for outcome in binary_outcomes:
        for group_variable in group_variables:
            contingency = pd.crosstab(
                frame[group_variable],
                frame[outcome],
            )

            if contingency.shape[0] < 2 or contingency.shape[1] < 2:
                chi2 = np.nan
                p_value = np.nan
                degrees_freedom = np.nan
                effect = np.nan
            else:
                (
                    chi2,
                    p_value,
                    degrees_freedom,
                    _,
                ) = chi2_contingency(
                    contingency,
                    correction=False,
                )
                effect = cramers_v(
                    contingency.to_numpy()
                )

            association_rows.append({
                "cohort": cohort_name,
                "outcome": outcome,
                "group_variable": group_variable,
                "n": int(len(frame)),
                "chi_square": chi2,
                "degrees_freedom": degrees_freedom,
                "p_value": p_value,
                "cramers_v": effect,
            })

association_tests = pd.DataFrame(
    association_rows
)

association_tests.to_csv(
    TABLE_DIR / "unknown_class_association_tests.csv",
    index=False,
)

print(
    association_tests.round(5).to_string(
        index=False
    )
)


## 7. Prepare the coordinate-valid holdout cohort

In [ ]:
valid_audit = (
    audit[
        audit["coordinate_valid"]
    ]
    .set_index("Mindat_id")
    .loc[
        [
            mindat_id
            for mindat_id in X_model.index
            if mindat_id
            in set(
                audit.loc[
                    audit["coordinate_valid"],
                    "Mindat_id",
                ]
            )
        ]
    ]
    .copy()
)

# Standardized alias used by the stronger-audit cells below.
valid_audit["continent"] = valid_audit["continent_country"]

valid_ids = valid_audit.index.tolist()
X_valid = X_model.loc[valid_ids].copy()

if X_valid.shape != (EXPECTED_VALID_N, EXPECTED_F):
    raise ValueError(
        f"Expected ({EXPECTED_VALID_N}, {EXPECTED_F}), "
        f"found {X_valid.shape}."
    )

valid_audit["Macrostrat_model_lithology"] = (
    valid_audit["lithology_A_class"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"evaporite": "other"})
)

valid_audit["age_category_model"] = (
    valid_audit["age_category"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

LITHOLOGY_LEVELS = sorted(
    valid_audit[
        "Macrostrat_model_lithology"
    ].unique()
)

lithology_to_index = {
    label: index
    for index, label in enumerate(
        LITHOLOGY_LEVELS
    )
}

age_to_index = {
    label: index
    for index, label in enumerate(
        AGE_LEVELS
    )
}

valid_audit["lithology_idx"] = (
    valid_audit[
        "Macrostrat_model_lithology"
    ].map(lithology_to_index)
)

valid_audit["age_idx"] = (
    valid_audit[
        "age_category_model"
    ].map(age_to_index)
)

if valid_audit[
    ["lithology_idx", "age_idx"]
].isna().any().any():
    raise ValueError(
        "Unmapped lithology or age categories."
    )

continents = valid_audit[
    "continent_country"
].to_numpy()

continent_counts = (
    valid_audit[
        "continent_country"
    ]
    .value_counts()
    .sort_index()
)

VIABLE_CONTINENTS = [
    continent
    for continent, count
    in continent_counts.items()
    if count >= MIN_HOLDOUT_N
    and continent != "Unknown"
]

if len(VIABLE_CONTINENTS) < 3:
    raise ValueError(
        "Fewer than three viable continents."
    )

print("Holdout continents:")
print(continent_counts.to_string())
print("\nViable:", VIABLE_CONTINENTS)
print(
    "\nPlanned full-mode fits:",
    len(VIABLE_CONTINENTS)
    * len(ACTIVE_SEEDS),
)


## 8. Fold preflight: category support and fixed vocabulary

In [ ]:
preflight_rows = []

for holdout_continent in VIABLE_CONTINENTS:
    test_mask = (
        valid_audit[
            "continent_country"
        ].eq(holdout_continent)
        .to_numpy()
    )
    train_mask = ~test_mask

    train_lithology = set(
        valid_audit.loc[
            train_mask,
            "Macrostrat_model_lithology",
        ]
    )
    test_lithology = set(
        valid_audit.loc[
            test_mask,
            "Macrostrat_model_lithology",
        ]
    )

    train_age = set(
        valid_audit.loc[
            train_mask,
            "age_category_model",
        ]
    )
    test_age = set(
        valid_audit.loc[
            test_mask,
            "age_category_model",
        ]
    )

    family_train_presence = (
        (X_valid.loc[train_mask] > 0)
        .sum(axis=0)
    )

    unsupported_test_mask = (
        ~valid_audit.loc[
            test_mask,
            "Macrostrat_model_lithology",
        ].isin(train_lithology)
        | ~valid_audit.loc[
            test_mask,
            "age_category_model",
        ].isin(train_age)
    )

    preflight_rows.append({
        "holdout_continent": holdout_continent,
        "n_train": int(train_mask.sum()),
        "n_test": int(test_mask.sum()),
        "training_lithology_levels": " | ".join(
            sorted(train_lithology)
        ),
        "test_lithology_levels": " | ".join(
            sorted(test_lithology)
        ),
        "unseen_test_lithology_levels": " | ".join(
            sorted(test_lithology - train_lithology)
        ),
        "training_age_levels": " | ".join(
            sorted(train_age)
        ),
        "test_age_levels": " | ".join(
            sorted(test_age)
        ),
        "unseen_test_age_levels": " | ".join(
            sorted(test_age - train_age)
        ),
        "n_test_with_unseen_covariate_level": int(
            unsupported_test_mask.sum()
        ),
        "minimum_family_training_presence": int(
            family_train_presence.min()
        ),
        "n_families_absent_from_training": int(
            (family_train_presence == 0).sum()
        ),
    })

preflight = pd.DataFrame(
    preflight_rows
)

preflight.to_csv(
    TABLE_DIR / "geographic_holdout_preflight.csv",
    index=False,
)

print(preflight.to_string(index=False))


## 9. Model definition

This is the original count-based sparse M7b architecture. Only the training-fold observations enter SVI.

In [ ]:
class HGCTM_Sparse(nn.Module):
    def __init__(
        self,
        K,
        F,
        L,
        A,
        sigma_mu=2.0,
        sigma_litho=0.30,
        sigma_age=0.15,
    ):
        super().__init__()
        self.K = K
        self.F = F
        self.L = L
        self.A = A
        self.sigma_mu = sigma_mu
        self.sigma_litho = sigma_litho
        self.sigma_age = sigma_age

    def model(
        self,
        X,
        lithology_idx,
        age_idx,
    ):
        N_local, F_local = X.shape
        K_local = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(
                torch.zeros(K_local, F_local),
                self.sigma_mu
                * torch.ones(K_local, F_local),
            ).to_event(2),
        )

        delta_litho = pyro.sample(
            "delta_litho",
            dist.Normal(
                torch.zeros(
                    K_local,
                    self.L,
                    F_local,
                ),
                self.sigma_litho
                * torch.ones(
                    K_local,
                    self.L,
                    F_local,
                ),
            ).to_event(3),
        )

        delta_age = pyro.sample(
            "delta_age",
            dist.Normal(
                torch.zeros(
                    K_local,
                    self.A,
                    F_local,
                ),
                self.sigma_age
                * torch.ones(
                    K_local,
                    self.A,
                    F_local,
                ),
            ).to_event(3),
        )

        omega_a = pyro.sample(
            "omega_a",
            dist.Beta(2.0, 2.0),
        )

        tau = pyro.sample(
            "tau",
            dist.Beta(
                torch.ones(F_local),
                3.0 * torch.ones(F_local),
            ).to_event(1),
        )

        B = pyro.sample(
            "B",
            dist.Normal(
                torch.zeros(
                    self.L + self.A,
                    K_local - 1,
                ),
                torch.ones(
                    self.L + self.A,
                    K_local - 1,
                ),
            ).to_event(2),
        )

        intercept = pyro.sample(
            "intercept",
            dist.Normal(
                torch.zeros(K_local - 1),
                torch.ones(K_local - 1),
            ).to_event(1),
        )

        log_kappa = pyro.sample(
            "log_kappa",
            dist.Normal(3.5, 1.0),
        )
        kappa = torch.exp(log_kappa)

        with pyro.plate("deposits", N_local):
            z_lithology = (
                torch.nn.functional.one_hot(
                    lithology_idx,
                    self.L,
                ).float()
            )

            z_age = (
                torch.nn.functional.one_hot(
                    age_idx,
                    self.A,
                ).float()
            )

            z = torch.cat(
                [z_lithology, z_age],
                dim=1,
            )

            eta = z @ B + intercept
            eta_full = torch.cat(
                [
                    eta,
                    torch.zeros(
                        N_local,
                        1,
                    ),
                ],
                dim=1,
            )
            theta = softmax(
                eta_full,
                dim=1,
            )

            lithology_deviation = (
                delta_litho[
                    :,
                    lithology_idx,
                    :,
                ].permute(1, 0, 2)
            )

            age_deviation = (
                delta_age[
                    :,
                    age_idx,
                    :,
                ].permute(1, 0, 2)
            )

            psi = (
                mu.unsqueeze(0)
                + tau.view(1, 1, -1)
                * lithology_deviation
                + omega_a
                * tau.view(1, 1, -1)
                * age_deviation
            )

            topic_family = softmax(
                psi,
                dim=2,
            )

            p = torch.einsum(
                "nk,nkf->nf",
                theta,
                topic_family,
            )

            p = p.clamp(min=1e-8)
            p = p / p.sum(
                dim=1,
                keepdim=True,
            )

            total_count = X.sum(
                dim=1
            ).int()

            pyro.sample(
                "obs",
                dist.DirichletMultinomial(
                    total_count=total_count,
                    concentration=kappa * p,
                ),
                obs=X,
            )

    def guide(
        self,
        X,
        lithology_idx,
        age_idx,
    ):
        _, F_local = X.shape
        K_local = self.K

        mu_loc = pyro.param(
            "mu_loc",
            torch.randn(
                K_local,
                F_local,
            ) * 0.1,
        )
        mu_scale = pyro.param(
            "mu_scale",
            0.5 * torch.ones(
                K_local,
                F_local,
            ),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "mu",
            dist.Normal(
                mu_loc,
                mu_scale,
            ).to_event(2),
        )

        delta_litho_loc = pyro.param(
            "delta_litho_loc",
            torch.zeros(
                K_local,
                self.L,
                F_local,
            ),
        )
        delta_litho_scale = pyro.param(
            "delta_litho_scale",
            0.2 * torch.ones(
                K_local,
                self.L,
                F_local,
            ),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_litho",
            dist.Normal(
                delta_litho_loc,
                delta_litho_scale,
            ).to_event(3),
        )

        delta_age_loc = pyro.param(
            "delta_age_loc",
            torch.zeros(
                K_local,
                self.A,
                F_local,
            ),
        )
        delta_age_scale = pyro.param(
            "delta_age_scale",
            0.1 * torch.ones(
                K_local,
                self.A,
                F_local,
            ),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_age",
            dist.Normal(
                delta_age_loc,
                delta_age_scale,
            ).to_event(3),
        )

        omega_a_a = pyro.param(
            "omega_a_a",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        omega_a_b = pyro.param(
            "omega_a_b",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "omega_a",
            dist.Beta(
                omega_a_a,
                omega_a_b,
            ),
        )

        tau_a = pyro.param(
            "tau_a",
            torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        tau_b = pyro.param(
            "tau_b",
            3.0 * torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "tau",
            dist.Beta(
                tau_a,
                tau_b,
            ).to_event(1),
        )

        B_loc = pyro.param(
            "B_loc",
            torch.zeros(
                self.L + self.A,
                K_local - 1,
            ),
        )
        B_scale = pyro.param(
            "B_scale",
            0.5 * torch.ones(
                self.L + self.A,
                K_local - 1,
            ),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "B",
            dist.Normal(
                B_loc,
                B_scale,
            ).to_event(2),
        )

        intercept_loc = pyro.param(
            "intercept_loc",
            torch.zeros(K_local - 1),
        )
        intercept_scale = pyro.param(
            "intercept_scale",
            0.5 * torch.ones(K_local - 1),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "intercept",
            dist.Normal(
                intercept_loc,
                intercept_scale,
            ).to_event(1),
        )

        log_kappa_loc = pyro.param(
            "log_kappa_loc",
            torch.tensor(3.5),
        )
        log_kappa_scale = pyro.param(
            "log_kappa_scale",
            torch.tensor(0.5),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "log_kappa",
            dist.Normal(
                log_kappa_loc,
                log_kappa_scale,
            ),
        )


## 10. Context-only Dirichlet–Multinomial baseline

This model uses the same lithology and age deviation structure, sparse family gate, age weight, and Dirichlet–Multinomial observation model, but has no latent topic mixture.

In [ ]:
class ContextOnlyDM(nn.Module):
    def __init__(
        self,
        F,
        L,
        A,
        sigma_mu=2.0,
        sigma_litho=0.30,
        sigma_age=0.15,
    ):
        super().__init__()
        self.F = F
        self.L = L
        self.A = A
        self.sigma_mu = sigma_mu
        self.sigma_litho = sigma_litho
        self.sigma_age = sigma_age

    def model(self, X, lithology_idx, age_idx):
        N_local, F_local = X.shape

        mu = pyro.sample(
            "mu",
            dist.Normal(
                torch.zeros(F_local),
                self.sigma_mu * torch.ones(F_local),
            ).to_event(1),
        )

        delta_litho = pyro.sample(
            "delta_litho",
            dist.Normal(
                torch.zeros(self.L, F_local),
                self.sigma_litho
                * torch.ones(self.L, F_local),
            ).to_event(2),
        )

        delta_age = pyro.sample(
            "delta_age",
            dist.Normal(
                torch.zeros(self.A, F_local),
                self.sigma_age
                * torch.ones(self.A, F_local),
            ).to_event(2),
        )

        omega_a = pyro.sample(
            "omega_a",
            dist.Beta(2.0, 2.0),
        )

        tau = pyro.sample(
            "tau",
            dist.Beta(
                torch.ones(F_local),
                3.0 * torch.ones(F_local),
            ).to_event(1),
        )

        log_kappa = pyro.sample(
            "log_kappa",
            dist.Normal(3.5, 1.0),
        )
        kappa = torch.exp(log_kappa)

        with pyro.plate("deposits", N_local):
            psi = (
                mu.unsqueeze(0)
                + tau.unsqueeze(0)
                * delta_litho[lithology_idx]
                + omega_a
                * tau.unsqueeze(0)
                * delta_age[age_idx]
            )

            p = softmax(psi, dim=1)

            pyro.sample(
                "obs",
                dist.DirichletMultinomial(
                    total_count=X.sum(dim=1).int(),
                    concentration=kappa * p,
                ),
                obs=X,
            )

    def guide(self, X, lithology_idx, age_idx):
        _, F_local = X.shape

        mu_loc = pyro.param(
            "mu_loc",
            torch.zeros(F_local),
        )
        mu_scale = pyro.param(
            "mu_scale",
            0.5 * torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "mu",
            dist.Normal(mu_loc, mu_scale).to_event(1),
        )

        delta_litho_loc = pyro.param(
            "delta_litho_loc",
            torch.zeros(self.L, F_local),
        )
        delta_litho_scale = pyro.param(
            "delta_litho_scale",
            0.2 * torch.ones(self.L, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_litho",
            dist.Normal(
                delta_litho_loc,
                delta_litho_scale,
            ).to_event(2),
        )

        delta_age_loc = pyro.param(
            "delta_age_loc",
            torch.zeros(self.A, F_local),
        )
        delta_age_scale = pyro.param(
            "delta_age_scale",
            0.1 * torch.ones(self.A, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_age",
            dist.Normal(
                delta_age_loc,
                delta_age_scale,
            ).to_event(2),
        )

        omega_a_a = pyro.param(
            "omega_a_a",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        omega_a_b = pyro.param(
            "omega_a_b",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "omega_a",
            dist.Beta(omega_a_a, omega_a_b),
        )

        tau_a = pyro.param(
            "tau_a",
            torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        tau_b = pyro.param(
            "tau_b",
            3.0 * torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "tau",
            dist.Beta(tau_a, tau_b).to_event(1),
        )

        log_kappa_loc = pyro.param(
            "log_kappa_loc",
            torch.tensor(3.5),
        )
        log_kappa_scale = pyro.param(
            "log_kappa_scale",
            torch.tensor(0.5),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "log_kappa",
            dist.Normal(
                log_kappa_loc,
                log_kappa_scale,
            ),
        )


## 11. Fixed fold-LDA, model fitting, prediction, and scoring utilities

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.set_rng_seed(seed)


def fit_fixed_training_lda(X_train, k=K):
    lda = LatentDirichletAllocation(
        n_components=k,
        doc_topic_prior=0.5 / k,
        topic_word_prior=0.01,
        max_iter=300,
        random_state=FIXED_LDA_SEED,
        learning_method="batch",
        n_jobs=-1,
    )
    lda.fit(X_train.astype(np.float64))
    phi = (
        lda.components_
        / lda.components_.sum(axis=1, keepdims=True)
    )
    return lda, phi


def beta_mean(name_a, name_b):
    a = float(pyro.param(name_a).item())
    b = float(pyro.param(name_b).item())
    return a / (a + b)


def train_svi(
    model,
    X_train,
    lithology_train,
    age_train,
    seed,
    mu_warm_start,
):
    set_all_seeds(seed)
    pyro.clear_param_store()

    svi = SVI(
        model.model,
        model.guide,
        ClippedAdam({
            "lr": LR,
            "betas": (0.9, 0.999),
        }),
        loss=Trace_ELBO(),
    )

    svi.step(
        X_train,
        lithology_train,
        age_train,
    )

    pyro.param("mu_loc").data.copy_(
        torch.tensor(
            mu_warm_start,
            dtype=torch.float32,
        )
    )

    pyro.param("tau_a").data.copy_(
        50.0 * torch.ones_like(
            pyro.param("tau_a")
        )
    )
    pyro.param("tau_b").data.copy_(
        torch.ones_like(
            pyro.param("tau_b")
        )
    )

    losses = []

    for step in range(N_STEPS):
        loss = svi.step(
            X_train,
            lithology_train,
            age_train,
        )
        losses.append(float(loss))

        if step < ANNEAL_END:
            progress = step / ANNEAL_END
            minimum_tau_a = (
                50.0 * (1.0 - progress)
                + progress
            )
            with torch.no_grad():
                pyro.param(
                    "tau_a"
                ).data.clamp_(
                    min=minimum_tau_a
                )

        if (step + 1) % PRINT_EVERY == 0:
            tau_a = pyro.param("tau_a").detach()
            tau_b = pyro.param("tau_b").detach()
            tau_mean = (
                tau_a
                / (tau_a + tau_b)
            ).mean().item()
            print(
                f"    step {step+1:>5d} "
                f"loss={loss:>12.1f} "
                f"tau_mean={tau_mean:.3f}"
            )

    return losses


def extract_hgctm_parameters():
    tau_a = pyro.param(
        "tau_a"
    ).detach().cpu().numpy()
    tau_b = pyro.param(
        "tau_b"
    ).detach().cpu().numpy()

    mu = pyro.param(
        "mu_loc"
    ).detach().cpu().numpy()

    phi_global = np.exp(
        mu - mu.max(
            axis=1,
            keepdims=True,
        )
    )
    phi_global /= phi_global.sum(
        axis=1,
        keepdims=True,
    )

    return {
        "mu": mu,
        "phi_global": phi_global,
        "delta_litho": pyro.param(
            "delta_litho_loc"
        ).detach().cpu().numpy(),
        "delta_age": pyro.param(
            "delta_age_loc"
        ).detach().cpu().numpy(),
        "B": pyro.param(
            "B_loc"
        ).detach().cpu().numpy(),
        "intercept": pyro.param(
            "intercept_loc"
        ).detach().cpu().numpy(),
        "tau": tau_a / (tau_a + tau_b),
        "omega_a": beta_mean(
            "omega_a_a",
            "omega_a_b",
        ),
        "kappa": float(
            np.exp(
                pyro.param(
                    "log_kappa_loc"
                ).item()
            )
        ),
    }


def extract_context_parameters():
    tau_a = pyro.param(
        "tau_a"
    ).detach().cpu().numpy()
    tau_b = pyro.param(
        "tau_b"
    ).detach().cpu().numpy()

    return {
        "mu": pyro.param(
            "mu_loc"
        ).detach().cpu().numpy(),
        "delta_litho": pyro.param(
            "delta_litho_loc"
        ).detach().cpu().numpy(),
        "delta_age": pyro.param(
            "delta_age_loc"
        ).detach().cpu().numpy(),
        "tau": tau_a / (tau_a + tau_b),
        "omega_a": beta_mean(
            "omega_a_a",
            "omega_a_b",
        ),
        "kappa": float(
            np.exp(
                pyro.param(
                    "log_kappa_loc"
                ).item()
            )
        ),
    }


In [ ]:
def predict_hgctm(
    params,
    lithology_idx,
    age_idx,
):
    lithology_tensor = torch.tensor(
        lithology_idx,
        dtype=torch.long,
    )
    age_tensor = torch.tensor(
        age_idx,
        dtype=torch.long,
    )

    mu = torch.tensor(
        params["mu"],
        dtype=torch.float32,
    )
    delta_litho = torch.tensor(
        params["delta_litho"],
        dtype=torch.float32,
    )
    delta_age = torch.tensor(
        params["delta_age"],
        dtype=torch.float32,
    )
    B = torch.tensor(
        params["B"],
        dtype=torch.float32,
    )
    intercept = torch.tensor(
        params["intercept"],
        dtype=torch.float32,
    )
    tau = torch.tensor(
        params["tau"],
        dtype=torch.float32,
    )
    omega_a = torch.tensor(
        params["omega_a"],
        dtype=torch.float32,
    )

    N_local = len(lithology_idx)

    with torch.no_grad():
        z_lithology = (
            torch.nn.functional.one_hot(
                lithology_tensor,
                len(LITHOLOGY_LEVELS),
            ).float()
        )
        z_age = (
            torch.nn.functional.one_hot(
                age_tensor,
                len(AGE_LEVELS),
            ).float()
        )
        z = torch.cat(
            [z_lithology, z_age],
            dim=1,
        )

        eta = z @ B + intercept
        eta_full = torch.cat(
            [
                eta,
                torch.zeros(N_local, 1),
            ],
            dim=1,
        )
        theta = softmax(
            eta_full,
            dim=1,
        )

        lithology_deviation = (
            delta_litho[
                :,
                lithology_tensor,
                :,
            ].permute(1, 0, 2)
        )
        age_deviation = (
            delta_age[
                :,
                age_tensor,
                :,
            ].permute(1, 0, 2)
        )

        psi = (
            mu.unsqueeze(0)
            + tau.view(1, 1, -1)
            * lithology_deviation
            + omega_a
            * tau.view(1, 1, -1)
            * age_deviation
        )

        topic_family = softmax(
            psi,
            dim=2,
        )
        p = torch.einsum(
            "nk,nkf->nf",
            theta,
            topic_family,
        )
        p = p.clamp(min=1e-10)
        p /= p.sum(
            dim=1,
            keepdim=True,
        )

    return {
        "theta": theta.cpu().numpy(),
        "topic_family": (
            topic_family.cpu().numpy()
        ),
        "p": p.cpu().numpy(),
    }


def predict_context_only(
    params,
    lithology_idx,
    age_idx,
):
    psi = (
        params["mu"][None, :]
        + params["tau"][None, :]
        * params["delta_litho"][
            lithology_idx
        ]
        + params["omega_a"]
        * params["tau"][None, :]
        * params["delta_age"][
            age_idx
        ]
    )

    psi -= psi.max(
        axis=1,
        keepdims=True,
    )

    p = np.exp(psi)
    p /= p.sum(
        axis=1,
        keepdims=True,
    )
    return p


def dm_log_prob_per_document(
    X,
    p,
    kappa,
):
    X = np.asarray(
        X,
        dtype=np.float64,
    )
    p = np.asarray(
        p,
        dtype=np.float64,
    )

    p = np.clip(
        p,
        1e-12,
        None,
    )
    p /= p.sum(
        axis=1,
        keepdims=True,
    )

    alpha = float(kappa) * p
    alpha0 = alpha.sum(axis=1)
    n_i = X.sum(axis=1)

    return (
        gammaln(n_i + 1.0)
        - gammaln(X + 1.0).sum(
            axis=1
        )
        + gammaln(alpha0)
        - gammaln(alpha0 + n_i)
        + (
            gammaln(alpha + X)
            - gammaln(alpha)
        ).sum(axis=1)
    )


def estimate_global_dm_baseline(
    X_train,
):
    counts = (
        X_train.sum(axis=0)
        + 0.5
    )
    p = counts / counts.sum()

    p_train = np.repeat(
        p[None, :],
        len(X_train),
        axis=0,
    )

    grid = np.array([
        0.5,
        1,
        2,
        5,
        10,
        20,
        50,
        100,
        200,
        500,
        1000,
        2000,
        5000,
        10000,
    ])

    rows = []

    for kappa in grid:
        ll = dm_log_prob_per_document(
            X_train,
            p_train,
            kappa,
        )
        rows.append({
            "kappa": float(kappa),
            "training_log_prob_total": float(
                ll.sum()
            ),
            "training_log_prob_per_count": float(
                ll.sum()
                / X_train.sum()
            ),
        })

    table = pd.DataFrame(rows)
    best_kappa = float(
        table.loc[
            table[
                "training_log_prob_total"
            ].idxmax(),
            "kappa",
        ]
    )

    return p, best_kappa, table


In [ ]:
def cosine_similarity(a, b):
    denominator = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )
    if denominator == 0:
        return 0.0
    return float(
        np.dot(a, b)
        / denominator
    )


def align_topics(
    run_profile,
    reference_profile,
):
    similarity = np.zeros(
        (
            run_profile.shape[0],
            reference_profile.shape[0],
        )
    )

    for i in range(
        run_profile.shape[0]
    ):
        for j in range(
            reference_profile.shape[0]
        ):
            similarity[i, j] = (
                cosine_similarity(
                    run_profile[i],
                    reference_profile[j],
                )
            )

    run_indices, reference_indices = (
        linear_sum_assignment(
            -similarity
        )
    )

    cosines = np.array([
        similarity[i, j]
        for i, j
        in zip(
            run_indices,
            reference_indices,
        )
    ])

    return {
        "run_indices": run_indices,
        "reference_indices": (
            reference_indices
        ),
        "matched_cosines": cosines,
        "mean_cosine": float(
            cosines.mean()
        ),
        "minimum_cosine": float(
            cosines.min()
        ),
    }


def paired_bootstrap(
    values_a,
    values_b,
    seed,
):
    differences = (
        np.asarray(
            values_a,
            dtype=float,
        )
        - np.asarray(
            values_b,
            dtype=float,
        )
    )

    rng = np.random.default_rng(seed)
    sampled = rng.choice(
        differences,
        size=(
            BOOTSTRAP_REPLICATES,
            len(differences),
        ),
        replace=True,
    )
    means = sampled.mean(axis=1)

    return {
        "mean_difference": float(
            differences.mean()
        ),
        "ci_low": float(
            np.quantile(
                means,
                0.025,
            )
        ),
        "ci_high": float(
            np.quantile(
                means,
                0.975,
            )
        ),
        "significantly_positive": bool(
            np.quantile(
                means,
                0.025,
            )
            > 0
        ),
    }


## 12. Full-cohort LDA and HGCTM references for post hoc topic retention

In [ ]:
X_valid_np = X_valid.to_numpy(
    dtype=np.float32
)

full_lda, phi_full_lda = (
    fit_fixed_training_lda(
        X_valid_np
    )
)

pd.DataFrame(
    phi_full_lda,
    columns=X_valid.columns,
).to_csv(
    TABLE_DIR
    / "full_coordinate_valid_lda_topics_k7.csv",
    index=False,
)


def load_full_hgctm_reference(seed):
    if FULL_REFERENCE_RUNS is None:
        return None

    path = (
        FULL_REFERENCE_RUNS
        / f"macrostrat__original__seed_{seed}.npz"
    )

    if not path.exists():
        return None

    archive = np.load(
        path,
        allow_pickle=True,
    )

    family_names = [
        str(value)
        for value
        in archive[
            "family_names"
        ].tolist()
    ]

    if family_names != [
        str(value)
        for value in X_valid.columns
    ]:
        raise ValueError(
            "Family order differs from full HGCTM reference."
        )

    return archive[
        "phi_global_aligned"
    ]


for seed in ACTIVE_SEEDS:
    print(
        "Full HGCTM reference seed",
        seed,
        "available:",
        load_full_hgctm_reference(
            seed
        )
        is not None,
    )


## 13. Optional nested training-only K audit

In [ ]:
def make_stratification_labels(
    frame,
):
    combined = (
        frame["continent"].astype(str)
        + " | "
        + frame["Deposit_type"].astype(str)
    )

    counts = combined.value_counts()
    labels = combined.where(
        ~combined.isin(
            set(
                counts[
                    counts < 2
                ].index
            )
        ),
        frame[
            "continent"
        ].astype(str),
    )

    counts = labels.value_counts()
    labels = labels.where(
        ~labels.isin(
            set(
                counts[
                    counts < 2
                ].index
            )
        ),
        "all",
    )

    return labels.to_numpy()


def nested_k_audit_fold(
    holdout_continent,
    train_mask,
):
    X_outer = X_valid_np[
        train_mask
    ]
    frame_outer = valid_audit.loc[
        train_mask
    ].copy()
    labels = make_stratification_labels(
        frame_outer
    )

    rows = []

    for k_candidate in K_CANDIDATES:
        reference_lda = (
            LatentDirichletAllocation(
                n_components=k_candidate,
                doc_topic_prior=(
                    0.5
                    / k_candidate
                ),
                topic_word_prior=0.01,
                max_iter=(
                    K_AUDIT_MAX_ITER
                ),
                random_state=(
                    FIXED_LDA_SEED
                ),
                learning_method="batch",
                n_jobs=-1,
            )
        )
        reference_lda.fit(
            X_outer.astype(
                np.float64
            )
        )

        phi_reference = (
            reference_lda.components_
            / reference_lda.components_.sum(
                axis=1,
                keepdims=True,
            )
        )

        splitter = (
            StratifiedShuffleSplit(
                n_splits=len(
                    K_AUDIT_SPLIT_SEEDS
                ),
                test_size=(
                    K_AUDIT_VALIDATION_FRACTION
                ),
                random_state=991,
            )
        )

        for split_number, (
            inner_train,
            inner_validation,
        ) in enumerate(
            splitter.split(
                X_outer,
                labels,
            ),
            start=1,
        ):
            lda = (
                LatentDirichletAllocation(
                    n_components=(
                        k_candidate
                    ),
                    doc_topic_prior=(
                        0.5
                        / k_candidate
                    ),
                    topic_word_prior=0.01,
                    max_iter=(
                        K_AUDIT_MAX_ITER
                    ),
                    random_state=(
                        FIXED_LDA_SEED
                    ),
                    learning_method="batch",
                    n_jobs=-1,
                )
            )
            lda.fit(
                X_outer[
                    inner_train
                ].astype(
                    np.float64
                )
            )

            phi_inner = (
                lda.components_
                / lda.components_.sum(
                    axis=1,
                    keepdims=True,
                )
            )

            alignment = align_topics(
                phi_inner,
                phi_reference,
            )

            rows.append({
                "holdout_continent": (
                    holdout_continent
                ),
                "K": int(
                    k_candidate
                ),
                "split_number": int(
                    split_number
                ),
                "n_inner_train": int(
                    len(inner_train)
                ),
                "n_inner_validation": int(
                    len(
                        inner_validation
                    )
                ),
                "validation_perplexity": float(
                    lda.perplexity(
                        X_outer[
                            inner_validation
                        ].astype(
                            np.float64
                        )
                    )
                ),
                "mean_topic_cosine": (
                    alignment[
                        "mean_cosine"
                    ]
                ),
                "minimum_topic_cosine": (
                    alignment[
                        "minimum_cosine"
                    ]
                ),
            })

    return pd.DataFrame(rows)


In [ ]:
continents_array = (
    valid_audit["continent"]
    .to_numpy()
)

nested_k_parts = []

if RUN_NESTED_K_AUDIT:
    for continent in VIABLE_CONTINENTS:
        path = (
            OUT
            / f"nested_k_audit__{continent.lower().replace(' ', '_')}.csv"
        )

        if path.exists():
            result = pd.read_csv(
                path
            )
        else:
            print(
                "Nested K audit:",
                continent,
            )
            train_mask = (
                continents_array
                != continent
            )
            result = (
                nested_k_audit_fold(
                    continent,
                    train_mask,
                )
            )
            result.to_csv(
                path,
                index=False,
            )

        nested_k_parts.append(
            result
        )

    nested_k_long = pd.concat(
        nested_k_parts,
        ignore_index=True,
    )

    nested_k_summary = (
        nested_k_long
        .groupby(
            [
                "holdout_continent",
                "K",
            ],
            as_index=False,
        )
        .agg(
            validation_perplexity_mean=(
                "validation_perplexity",
                "mean",
            ),
            validation_perplexity_std=(
                "validation_perplexity",
                "std",
            ),
            mean_topic_cosine_mean=(
                "mean_topic_cosine",
                "mean",
            ),
            minimum_topic_cosine_min=(
                "minimum_topic_cosine",
                "min",
            ),
        )
    )

    nested_k_summary[
        "perplexity_rank"
    ] = (
        nested_k_summary
        .groupby(
            "holdout_continent"
        )[
            "validation_perplexity_mean"
        ]
        .rank(
            method="min",
            ascending=True,
        )
    )

    nested_k_summary[
        "stability_rank"
    ] = (
        nested_k_summary
        .groupby(
            "holdout_continent"
        )[
            "mean_topic_cosine_mean"
        ]
        .rank(
            method="min",
            ascending=False,
        )
    )

    nested_k_summary[
        "combined_rank"
    ] = (
        nested_k_summary[
            "perplexity_rank"
        ]
        + nested_k_summary[
            "stability_rank"
        ]
    )

    nested_k_long.to_csv(
        TABLE_DIR
        / "nested_training_only_k_audit_long.csv",
        index=False,
    )
    nested_k_summary.to_csv(
        TABLE_DIR
        / "nested_training_only_k_audit_summary.csv",
        index=False,
    )

    print(
        nested_k_summary.round(4).to_string(
            index=False
        )
    )
else:
    nested_k_long = pd.DataFrame()
    nested_k_summary = pd.DataFrame()
    print("Nested K audit disabled.")


## 14. Resumable run helpers and fixed fold-specific LDA warm starts

In [ ]:
def safe_name(text):
    return (
        text.lower()
        .replace(" ", "_")
    )


def run_stem(
    model_name,
    continent,
    seed,
):
    return (
        f"{model_name}"
        f"__{safe_name(continent)}"
        f"__seed_{seed}"
    )


def load_summary(
    model_name,
    continent,
    seed,
):
    path = (
        RUNS_DIR
        / f"{run_stem(model_name, continent, seed)}"
        "_run_summary.json"
    )
    if not path.exists():
        return None
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


fold_lda_topics = {}
fold_lda_rows = []

for continent in VIABLE_CONTINENTS:
    train_mask = (
        continents_array
        != continent
    )

    _, phi_fold = (
        fit_fixed_training_lda(
            X_valid_np[
                train_mask
            ]
        )
    )

    fold_lda_topics[
        continent
    ] = phi_fold

    alignment = align_topics(
        phi_fold,
        phi_full_lda,
    )

    fold_lda_rows.append({
        "holdout_continent": (
            continent
        ),
        "fixed_lda_seed": (
            FIXED_LDA_SEED
        ),
        "mean_fold_lda_to_full_lda_cosine": (
            alignment[
                "mean_cosine"
            ]
        ),
        "minimum_fold_lda_to_full_lda_cosine": (
            alignment[
                "minimum_cosine"
            ]
        ),
    })

fold_lda_summary = pd.DataFrame(
    fold_lda_rows
)

fold_lda_summary.to_csv(
    TABLE_DIR
    / "fold_lda_topic_retention_to_full_lda.csv",
    index=False,
)

print(
    fold_lda_summary.round(4).to_string(
        index=False
    )
)


## 15. Run HGCTM, context-only DM, and global DM

In [ ]:
lithology_idx_all = (
    valid_audit[
        "lithology_idx"
    ].to_numpy(
        dtype=np.int64
    )
)
age_idx_all = (
    valid_audit[
        "age_idx"
    ].to_numpy(
        dtype=np.int64
    )
)

comparison_rows = []

for continent in VIABLE_CONTINENTS:
    test_mask = (
        continents_array
        == continent
    )
    train_mask = ~test_mask

    X_train = X_valid_np[
        train_mask
    ]
    X_test = X_valid_np[
        test_mask
    ]

    lithology_train = (
        lithology_idx_all[
            train_mask
        ]
    )
    lithology_test = (
        lithology_idx_all[
            test_mask
        ]
    )
    age_train = (
        age_idx_all[
            train_mask
        ]
    )
    age_test = (
        age_idx_all[
            test_mask
        ]
    )

    train_ids = np.array(
        valid_ids
    )[train_mask]
    test_ids = np.array(
        valid_ids
    )[test_mask]

    phi_fold_lda = (
        fold_lda_topics[
            continent
        ]
    )

    mu_hgctm_warm = np.log(
        np.clip(
            phi_fold_lda,
            1e-8,
            None,
        )
    )
    mu_hgctm_warm -= (
        mu_hgctm_warm.mean(
            axis=1,
            keepdims=True,
        )
    )

    training_composition = (
        X_train.sum(axis=0)
        + 0.5
    )
    training_composition /= (
        training_composition.sum()
    )
    mu_context_warm = np.log(
        np.clip(
            training_composition,
            1e-8,
            None,
        )
    )
    mu_context_warm -= (
        mu_context_warm.mean()
    )

    (
        global_p,
        global_kappa,
        global_grid,
    ) = estimate_global_dm_baseline(
        X_train
    )

    global_grid.to_csv(
        RUNS_DIR
        / f"global_dm__{safe_name(continent)}_kappa_grid.csv",
        index=False,
    )

    global_p_test = np.repeat(
        global_p[None, :],
        len(X_test),
        axis=0,
    )
    global_ll = (
        dm_log_prob_per_document(
            X_test,
            global_p_test,
            global_kappa,
        )
    )

    for seed in ACTIVE_SEEDS:
        print("\n" + "=" * 90)
        print(
            f"HOLDOUT={continent} | "
            f"N_train={len(X_train)} | "
            f"N_test={len(X_test)} | "
            f"PYRO_SEED={seed} | "
            f"FIXED_LDA_SEED={FIXED_LDA_SEED}"
        )
        print("=" * 90)

        # ---------- HGCTM ----------
        hgctm_summary = load_summary(
            "strong_hgctm",
            continent,
            seed,
        )

        hgctm_stem = run_stem(
            "strong_hgctm",
            continent,
            seed,
        )

        if hgctm_summary is None:
            model = HGCTM_Sparse(
                K=K,
                F=EXPECTED_F,
                L=len(
                    LITHOLOGY_LEVELS
                ),
                A=len(
                    AGE_LEVELS
                ),
                sigma_mu=SIGMA_MU,
                sigma_litho=(
                    SIGMA_LITHO
                ),
                sigma_age=(
                    SIGMA_AGE
                ),
            )

            losses = train_svi(
                model,
                torch.tensor(
                    X_train,
                    dtype=torch.float32,
                ),
                torch.tensor(
                    lithology_train,
                    dtype=torch.long,
                ),
                torch.tensor(
                    age_train,
                    dtype=torch.long,
                ),
                seed,
                mu_hgctm_warm,
            )

            params = (
                extract_hgctm_parameters()
            )
            prediction = (
                predict_hgctm(
                    params,
                    lithology_test,
                    age_test,
                )
            )
            hgctm_ll = (
                dm_log_prob_per_document(
                    X_test,
                    prediction["p"],
                    params["kappa"],
                )
            )

            alignment_fold_lda = (
                align_topics(
                    params[
                        "phi_global"
                    ],
                    phi_fold_lda,
                )
            )

            full_reference = (
                load_full_hgctm_reference(
                    seed
                )
            )

            if full_reference is None:
                alignment_full = None
            else:
                alignment_full = (
                    align_topics(
                        params[
                            "phi_global"
                        ],
                        full_reference,
                    )
                )

            hgctm_summary = {
                "model": (
                    "strong_hgctm"
                ),
                "holdout_continent": (
                    continent
                ),
                "seed": int(seed),
                "fixed_lda_seed": int(
                    FIXED_LDA_SEED
                ),
                "n_train": int(
                    len(X_train)
                ),
                "n_test": int(
                    len(X_test)
                ),
                "test_log_prob_per_recorded_count": float(
                    hgctm_ll.sum()
                    / X_test.sum()
                ),
                "test_log_prob_per_deposit": float(
                    hgctm_ll.mean()
                ),
                "kappa": float(
                    params["kappa"]
                ),
                "omega_a": float(
                    params["omega_a"]
                ),
                "mean_topic_cosine_to_fold_lda": (
                    alignment_fold_lda[
                        "mean_cosine"
                    ]
                ),
                "minimum_topic_cosine_to_fold_lda": (
                    alignment_fold_lda[
                        "minimum_cosine"
                    ]
                ),
                "mean_topic_cosine_to_full_hgctm_posthoc": (
                    None
                    if alignment_full
                    is None
                    else alignment_full[
                        "mean_cosine"
                    ]
                ),
                "minimum_topic_cosine_to_full_hgctm_posthoc": (
                    None
                    if alignment_full
                    is None
                    else alignment_full[
                        "minimum_cosine"
                    ]
                ),
            }

            np.savez_compressed(
                RUNS_DIR
                / f"{hgctm_stem}.npz",
                train_Mindat_id=(
                    train_ids
                ),
                test_Mindat_id=(
                    test_ids
                ),
                phi_fold_lda=(
                    phi_fold_lda
                ),
                phi_global=params[
                    "phi_global"
                ],
                predicted_theta=(
                    prediction["theta"]
                ),
                predicted_family_probabilities=(
                    prediction["p"]
                ),
                test_log_prob_per_document=(
                    hgctm_ll
                ),
                losses=np.asarray(
                    losses
                ),
            )

            with open(
                RUNS_DIR
                / f"{hgctm_stem}_run_summary.json",
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    hgctm_summary,
                    handle,
                    indent=2,
                )
        else:
            archive = np.load(
                RUNS_DIR
                / f"{hgctm_stem}.npz",
                allow_pickle=True,
            )
            hgctm_ll = archive[
                "test_log_prob_per_document"
            ]

        # ---------- Context-only DM ----------
        context_summary = load_summary(
            "context_only_dm",
            continent,
            seed,
        )

        context_stem = run_stem(
            "context_only_dm",
            continent,
            seed,
        )

        if context_summary is None:
            context_model = (
                ContextOnlyDM(
                    F=EXPECTED_F,
                    L=len(
                        LITHOLOGY_LEVELS
                    ),
                    A=len(
                        AGE_LEVELS
                    ),
                    sigma_mu=SIGMA_MU,
                    sigma_litho=(
                        SIGMA_LITHO
                    ),
                    sigma_age=(
                        SIGMA_AGE
                    ),
                )
            )

            context_losses = (
                train_svi(
                    context_model,
                    torch.tensor(
                        X_train,
                        dtype=(
                            torch.float32
                        ),
                    ),
                    torch.tensor(
                        lithology_train,
                        dtype=(
                            torch.long
                        ),
                    ),
                    torch.tensor(
                        age_train,
                        dtype=(
                            torch.long
                        ),
                    ),
                    seed,
                    mu_context_warm,
                )
            )

            context_params = (
                extract_context_parameters()
            )
            context_p = (
                predict_context_only(
                    context_params,
                    lithology_test,
                    age_test,
                )
            )
            context_ll = (
                dm_log_prob_per_document(
                    X_test,
                    context_p,
                    context_params[
                        "kappa"
                    ],
                )
            )

            context_summary = {
                "model": (
                    "context_only_dm"
                ),
                "holdout_continent": (
                    continent
                ),
                "seed": int(seed),
                "n_train": int(
                    len(X_train)
                ),
                "n_test": int(
                    len(X_test)
                ),
                "test_log_prob_per_recorded_count": float(
                    context_ll.sum()
                    / X_test.sum()
                ),
                "test_log_prob_per_deposit": float(
                    context_ll.mean()
                ),
                "kappa": float(
                    context_params[
                        "kappa"
                    ]
                ),
                "omega_a": float(
                    context_params[
                        "omega_a"
                    ]
                ),
            }

            np.savez_compressed(
                RUNS_DIR
                / f"{context_stem}.npz",
                train_Mindat_id=(
                    train_ids
                ),
                test_Mindat_id=(
                    test_ids
                ),
                predicted_family_probabilities=(
                    context_p
                ),
                test_log_prob_per_document=(
                    context_ll
                ),
                losses=np.asarray(
                    context_losses
                ),
            )

            with open(
                RUNS_DIR
                / f"{context_stem}_run_summary.json",
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    context_summary,
                    handle,
                    indent=2,
                )
        else:
            archive = np.load(
                RUNS_DIR
                / f"{context_stem}.npz",
                allow_pickle=True,
            )
            context_ll = archive[
                "test_log_prob_per_document"
            ]

        hgctm_vs_context = (
            paired_bootstrap(
                hgctm_ll,
                context_ll,
                seed + 1000,
            )
        )
        context_vs_global = (
            paired_bootstrap(
                context_ll,
                global_ll,
                seed + 2000,
            )
        )
        hgctm_vs_global = (
            paired_bootstrap(
                hgctm_ll,
                global_ll,
                seed + 3000,
            )
        )

        lda_row = (
            fold_lda_summary[
                fold_lda_summary[
                    "holdout_continent"
                ].eq(continent)
            ].iloc[0]
        )

        comparison_rows.append({
            "holdout_continent": (
                continent
            ),
            "seed": int(seed),
            "n_train": int(
                len(X_train)
            ),
            "n_test": int(
                len(X_test)
            ),
            "hgctm_log_prob_per_recorded_count": float(
                hgctm_ll.sum()
                / X_test.sum()
            ),
            "context_only_log_prob_per_recorded_count": float(
                context_ll.sum()
                / X_test.sum()
            ),
            "global_dm_log_prob_per_recorded_count": float(
                global_ll.sum()
                / X_test.sum()
            ),
            "hgctm_minus_context_mean_per_deposit": (
                hgctm_vs_context[
                    "mean_difference"
                ]
            ),
            "hgctm_minus_context_ci_low": (
                hgctm_vs_context[
                    "ci_low"
                ]
            ),
            "hgctm_minus_context_ci_high": (
                hgctm_vs_context[
                    "ci_high"
                ]
            ),
            "hgctm_beats_context_ci": (
                hgctm_vs_context[
                    "significantly_positive"
                ]
            ),
            "context_minus_global_mean_per_deposit": (
                context_vs_global[
                    "mean_difference"
                ]
            ),
            "context_beats_global_ci": (
                context_vs_global[
                    "significantly_positive"
                ]
            ),
            "hgctm_minus_global_mean_per_deposit": (
                hgctm_vs_global[
                    "mean_difference"
                ]
            ),
            "hgctm_beats_global_ci": (
                hgctm_vs_global[
                    "significantly_positive"
                ]
            ),
            "fold_lda_mean_cosine_to_full_lda": float(
                lda_row[
                    "mean_fold_lda_to_full_lda_cosine"
                ]
            ),
            "fold_lda_minimum_cosine_to_full_lda": float(
                lda_row[
                    "minimum_fold_lda_to_full_lda_cosine"
                ]
            ),
            "hgctm_mean_cosine_to_full_hgctm_posthoc": (
                hgctm_summary[
                    "mean_topic_cosine_to_full_hgctm_posthoc"
                ]
            ),
            "hgctm_minimum_cosine_to_full_hgctm_posthoc": (
                hgctm_summary[
                    "minimum_topic_cosine_to_full_hgctm_posthoc"
                ]
            ),
            "hgctm_mean_cosine_to_fixed_fold_lda": (
                hgctm_summary[
                    "mean_topic_cosine_to_fold_lda"
                ]
            ),
        })

        print(
            f"  HGCTM={hgctm_ll.sum()/X_test.sum():.4f}; "
            f"Context={context_ll.sum()/X_test.sum():.4f}; "
            f"Global={global_ll.sum()/X_test.sum():.4f}; "
            f"HGCTM>Context={hgctm_vs_context['significantly_positive']}"
        )

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df.to_csv(
    TABLE_DIR
    / "strong_geographic_predictive_comparisons_all_runs.csv",
    index=False,
)

print("Completed comparisons:", len(comparison_df))


## 16. Aggregate across seeds and evaluate HGCTM topic stability

In [ ]:
metric_columns = [
    "hgctm_log_prob_per_recorded_count",
    "context_only_log_prob_per_recorded_count",
    "global_dm_log_prob_per_recorded_count",
    "hgctm_minus_context_mean_per_deposit",
    "context_minus_global_mean_per_deposit",
    "hgctm_minus_global_mean_per_deposit",
    "fold_lda_mean_cosine_to_full_lda",
    "fold_lda_minimum_cosine_to_full_lda",
    "hgctm_mean_cosine_to_full_hgctm_posthoc",
    "hgctm_minimum_cosine_to_full_hgctm_posthoc",
    "hgctm_mean_cosine_to_fixed_fold_lda",
]

aggregate_rows = []

for continent, group in (
    comparison_df.groupby(
        "holdout_continent",
        sort=True,
    )
):
    row = {
        "holdout_continent": (
            continent
        ),
        "n_seeds": int(
            len(group)
        ),
        "n_test": int(
            group["n_test"].iloc[0]
        ),
        "hgctm_beats_context_all_seeds": bool(
            group[
                "hgctm_beats_context_ci"
            ].all()
        ),
        "context_beats_global_all_seeds": bool(
            group[
                "context_beats_global_ci"
            ].all()
        ),
        "hgctm_beats_global_all_seeds": bool(
            group[
                "hgctm_beats_global_ci"
            ].all()
        ),
    }

    for column in metric_columns:
        values = pd.to_numeric(
            group[column],
            errors="coerce",
        )
        row[
            f"{column}_median"
        ] = float(
            values.median()
        )
        row[
            f"{column}_min"
        ] = float(
            values.min()
        )
        row[
            f"{column}_max"
        ] = float(
            values.max()
        )
        row[
            f"{column}_std"
        ] = float(
            values.std(ddof=0)
        )

    aggregate_rows.append(row)

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR
    / "strong_geographic_summary_across_seeds.csv",
    index=False,
)

seed_stability_rows = []

for continent in VIABLE_CONTINENTS:
    reference_seed = (
        PRIMARY_SEED
        if PRIMARY_SEED
        in ACTIVE_SEEDS
        else ACTIVE_SEEDS[0]
    )

    reference = np.load(
        RUNS_DIR
        / f"{run_stem('strong_hgctm', continent, reference_seed)}.npz",
        allow_pickle=True,
    )["phi_global"]

    for seed in ACTIVE_SEEDS:
        profile = np.load(
            RUNS_DIR
            / f"{run_stem('strong_hgctm', continent, seed)}.npz",
            allow_pickle=True,
        )["phi_global"]

        alignment = align_topics(
            profile,
            reference,
        )

        for cosine in (
            alignment[
                "matched_cosines"
            ]
        ):
            seed_stability_rows.append({
                "holdout_continent": (
                    continent
                ),
                "reference_seed": (
                    reference_seed
                ),
                "seed": seed,
                "topic_cosine": float(
                    cosine
                ),
            })

seed_stability = pd.DataFrame(
    seed_stability_rows
)

seed_stability.to_csv(
    TABLE_DIR
    / "strong_geographic_hgctm_seed_stability.csv",
    index=False,
)

print(
    aggregate_df.round(4).to_string(
        index=False
    )
)

print("\nSeed stability:")
print(
    seed_stability.groupby(
        "holdout_continent"
    )[
        "topic_cosine"
    ].agg(
        ["mean", "min", "std"]
    ).round(4).to_string()
)


## 17. Reviewer-oriented decision table

In [ ]:
decision_rows = []

for _, row in (
    aggregate_df.iterrows()
):
    beats_context = bool(
        row[
            "hgctm_beats_context_all_seeds"
        ]
    )
    beats_global = bool(
        row[
            "hgctm_beats_global_all_seeds"
        ]
    )

    topic_retention = row[
        "hgctm_mean_cosine_to_full_hgctm_posthoc_median"
    ]

    if (
        beats_context
        and beats_global
        and (
            not np.isfinite(
                topic_retention
            )
            or topic_retention >= 0.80
        )
    ):
        conclusion = (
            "strong_topic_layer_geographic_support"
        )
    elif beats_context:
        conclusion = (
            "topic_layer_adds_prediction_but_topic_retention_is_partial"
        )
    elif beats_global:
        conclusion = (
            "hgctm_beats_global_but_not_context_only"
        )
    else:
        conclusion = (
            "no_consistent_geographic_predictive_gain"
        )

    decision_rows.append({
        "holdout_continent": (
            row[
                "holdout_continent"
            ]
        ),
        "n_test": int(
            row["n_test"]
        ),
        "hgctm_beats_context_all_seeds": (
            beats_context
        ),
        "context_beats_global_all_seeds": bool(
            row[
                "context_beats_global_all_seeds"
            ]
        ),
        "hgctm_beats_global_all_seeds": (
            beats_global
        ),
        "median_hgctm_minus_context_per_deposit": (
            row[
                "hgctm_minus_context_mean_per_deposit_median"
            ]
        ),
        "median_fold_lda_to_full_lda_cosine": (
            row[
                "fold_lda_mean_cosine_to_full_lda_median"
            ]
        ),
        "median_hgctm_to_full_hgctm_cosine": (
            topic_retention
        ),
        "conclusion": conclusion,
    })

decision_table = pd.DataFrame(
    decision_rows
)

decision_table.to_csv(
    TABLE_DIR
    / "reviewer_strong_geographic_decision_table.csv",
    index=False,
)

print(
    decision_table.round(4).to_string(
        index=False
    )
)


## 18. Figures

In [ ]:
positions = np.arange(
    len(aggregate_df)
)

fig, ax = plt.subplots(
    figsize=(11, 5.5)
)

ax.scatter(
    positions,
    aggregate_df[
        "hgctm_log_prob_per_recorded_count_median"
    ],
    label="HGCTM",
)

ax.scatter(
    positions,
    aggregate_df[
        "context_only_log_prob_per_recorded_count_median"
    ],
    label="Context-only DM",
)

ax.scatter(
    positions,
    aggregate_df[
        "global_dm_log_prob_per_recorded_count_median"
    ],
    label="Global DM",
)

ax.set_xticks(positions)
ax.set_xticklabels(
    aggregate_df[
        "holdout_continent"
    ],
    rotation=30,
    ha="right",
)

ax.set_ylabel(
    "Held-out DM log probability per recorded count"
)
ax.set_title(
    "Directly comparable geographic holdout models"
)
ax.legend()
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "strong_geographic_predictive_scores.png",
    dpi=300,
)
plt.close()


fig, ax = plt.subplots(
    figsize=(11, 5.5)
)

ax.scatter(
    positions,
    aggregate_df[
        "fold_lda_mean_cosine_to_full_lda_median"
    ],
    label="Fold LDA → full LDA",
)

ax.scatter(
    positions,
    aggregate_df[
        "hgctm_mean_cosine_to_full_hgctm_posthoc_median"
    ],
    label="Fold HGCTM → full HGCTM",
)

ax.set_ylim(0, 1.02)
ax.set_xticks(positions)
ax.set_xticklabels(
    aggregate_df[
        "holdout_continent"
    ],
    rotation=30,
    ha="right",
)

ax.set_ylabel(
    "Mean aligned-topic cosine"
)
ax.set_title(
    "Post hoc topic retention"
)
ax.legend()
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "strong_geographic_topic_retention.png",
    dpi=300,
)
plt.close()


fig, ax = plt.subplots(
    figsize=(11, 5.5)
)

ax.scatter(
    positions,
    aggregate_df[
        "hgctm_minus_context_mean_per_deposit_median"
    ],
)

ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(positions)
ax.set_xticklabels(
    aggregate_df[
        "holdout_continent"
    ],
    rotation=30,
    ha="right",
)

ax.set_ylabel(
    "HGCTM − context-only mean log probability per deposit"
)
ax.set_title(
    "Does the topic layer add beyond lithology and age?"
)
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "hgctm_minus_context_only.png",
    dpi=300,
)
plt.close()


if (
    RUN_NESTED_K_AUDIT
    and len(
        nested_k_summary
    )
):
    fig, ax = plt.subplots(
        figsize=(11, 5.5)
    )

    for continent in VIABLE_CONTINENTS:
        subset = (
            nested_k_summary[
                nested_k_summary[
                    "holdout_continent"
                ].eq(continent)
            ]
        )

        ax.plot(
            subset["K"],
            subset[
                "validation_perplexity_mean"
            ],
            marker="o",
            label=continent,
        )

    ax.set_xlabel("K")
    ax.set_ylabel(
        "Inner-validation LDA perplexity"
    )
    ax.set_title(
        "Training-only K audit"
    )
    ax.legend(
        ncol=2,
        fontsize=8,
    )
    plt.tight_layout()
    plt.savefig(
        FIG_DIR
        / "nested_training_only_k_audit.png",
        dpi=300,
    )
    plt.close()

print("Saved figures:")
for path in sorted(
    FIG_DIR.iterdir()
):
    print(" -", path.name)


## 19. Provenance and output archive

In [ ]:
manifest = {
    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "analysis": (
        "strong_geographic_generalisation_audit"
    ),
    "notebook": (
        "HGCTM_Strong_Geographic_Generalisation_Audit_K7.ipynb"
    ),
    "cohort": {
        "full_model_set": int(
            len(audit)
        ),
        "coordinate_valid": int(
            len(valid_audit)
        ),
        "excluded_placeholder_coordinates": int(
            (
                ~audit[
                    "coordinate_valid"
                ]
            ).sum()
        ),
    },
    "outer_holdout": {
        "continents": (
            VIABLE_CONTINENTS
        ),
        "K": K,
        "pyro_seeds": (
            ACTIVE_SEEDS
        ),
        "fixed_fold_lda_seed": (
            FIXED_LDA_SEED
        ),
    },
    "models": {
        "hgctm": (
            "Sparse M7b with fixed training-only LDA warm start."
        ),
        "context_only_dm": (
            "Lithology- and age-conditioned DM without topics."
        ),
        "global_dm": (
            "Training-only global DM."
        ),
    },
    "predictive_scoring": (
        "All predictive models use the same complete held-out "
        "Dirichlet-Multinomial likelihood."
    ),
    "nested_k_audit": {
        "enabled": (
            RUN_NESTED_K_AUDIT
        ),
        "candidates": (
            K_CANDIDATES
        ),
        "interpretation": (
            "Training-only LDA audit, not full nested HGCTM selection."
        ),
    },
    "limitation": (
        "Post-selection geographic sensitivity analysis; "
        "not untouched external model selection."
    ),
}

with open(
    OUT / "run_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

readme = '''
HGCTM strong geographic generalisation audit

- Fixed training-only LDA warm start per continent.
- HGCTM compared with context-only DM and global DM using the same
  complete held-out Dirichlet-Multinomial likelihood.
- LDA used for topic retention, not raw likelihood comparison.
- Optional training-only K=5..9 audit.
- Results must be described as post-selection geographic sensitivity,
  not untouched external model selection.
'''.strip()

(
    OUT / "README.txt"
).write_text(
    readme,
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(
        WORK
        / "HGCTM_Strong_Geographic_Generalisation_Audit_K7_Outputs"
    ),
    "zip",
    root_dir=OUT,
)

print("Output archive:")
print(archive_path)
